In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
import re
from pathlib import Path
from typing import List, Dict, Any

In [ ]:
from bs4 import BeautifulSoup
import google.generativeai as genai
from pymilvus import (
    FieldSchema, CollectionSchema, DataType, Collection,
    connections, utility
)

### Step 0: Index document into vector database

In [ ]:
def extract_text_from_html_file(path: str) -> str:
    html = Path(path).read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "lxml")
    # Remove script/style
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    text = soup.get_text(" ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def chunk_text(text: str, chunk_size: int = 800, overlap: int = 120) -> List[str]:
    # simple char-based chunking
    chunks = []
    i = 0
    while i < len(text):
        chunk = text[i:i+chunk_size]
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks


def embed_texts(texts: List[str], model: str = "text-embedding-004") -> List[List[float]]:
    genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
    # The google-generativeai SDK supports batching via embed_content
    # We'll call for each text for simplicity; for large corpora, batch your calls.
    embeddings = []
    for t in texts:
        resp = genai.embed_content(model=model, content=t)
        vec = resp["embedding"] if isinstance(resp, dict) else resp.embedding
        embeddings.append(vec)
    return embeddings

In [ ]:
# 1) Connect
connections.connect(alias="default", host="127.0.0.1", port="19530")
print("Milvus connected:", connections.has_connection("default"))

In [ ]:
# 2) Prepare data
DATA_DIR = Path("downloaded_html")
files = sorted([str(p) for p in DATA_DIR.glob("*.html")])
print("Files:", files)

records = []  # each: {source, chunk_id, text}
for f in files:
    txt = extract_text_from_html_file(f)
    chunks = chunk_text(txt, chunk_size=800, overlap=120)
    for i, ch in enumerate(chunks):
        records.append({"source": Path(f).name, "chunk_id": i, "text": ch})

texts = [r["text"] for r in records]
embs = embed_texts(texts)

In [ ]:
# 3) Define schema
collection_name = "rag_chunks"
if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)

fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name="chunk_id", dtype=DataType.INT64),
    FieldSchema(name="source", dtype=DataType.VARCHAR, max_length=512),
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
    FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=len(embs[0])),
]

schema = CollectionSchema(fields=fields, description="RAG demo chunks")
col = Collection(name=collection_name, schema=schema)

In [ ]:
# 4) Insert data
entities = [
    [r["chunk_id"] for r in records],
    [r["source"] for r in records],
    [r["text"] for r in records],
    embs,
]
# Because id is auto_id, we omit it.
col.insert(entities)
col.flush()
print("Inserted:", col.num_entities)

In [ ]:
# 5) Create index and load
index_params = {
    "index_type": "HNSW",
    "metric_type": "COSINE",
    "params": {"M": 8, "efConstruction": 64}
}
col.create_index(field_name="embedding", index_params=index_params)
col.load()
print("Collection loaded.")

### Step 1: Retrieve

In [ ]:
def milvus_retrieve(question: str, top_k: int = 4) -> List[Dict]:
    q_emb = embed_texts([question])[0]
    search_params = {"metric_type": "COSINE", "params": {"ef": 64}}
    results = col.search(
        data=[q_emb],
        anns_field="embedding",
        param=search_params,
        limit=top_k,
        output_fields=["source", "chunk_id", "text"],
    )
    out = []
    for hits in results:
        for hit in hits:
            out.append({
                "score": float(hit.distance),
                "source": hit.entity.get("source"),
                "chunk_id": int(hit.entity.get("chunk_id")),
                "text": hit.entity.get("text"),
            })
    return out

In [ ]:
QUESTION = ""

In [ ]:
retrieved_results = milvus_retrieve(QUESTION, top_k=4)
retrieved_results


### Step 2: Augment Prompt

In [ ]:
def build_augmented_prompt(question: str, retrieved_chunks: List[Dict[str, Any]]) -> str:
    context_blocks = [f"[Source: {r['source']} | Chunk: {r['chunk_id']}]{r['text']}" for r in retrieved_chunks]
    context_text = "".join(context_blocks)
    system_instructions = (
        "You are a helpful assistant. Use only the given context to answer. "
        "If the answer is not in the context, say you don't know and suggest next steps."
    )
    prompt = f"""
        {system_instructions}
        Context: {context_text}
        Question: {question}
        Answer concisely with citations like [source:chunk].
    """
    return prompt

In [ ]:
rag_prompt = build_augmented_prompt(QUESTION, retrieved_results)
rag_prompt


### Step 3: Generate response

In [ ]:
def generate_with_gemini(prompt: str, model: str = "gemini-1.5-pro") -> str:
    genai.configure(api_key=os.environ.get("GOOGLE_API_KEY"))
    m = genai.GenerativeModel(model)
    resp = m.generate_content(prompt)
    # google-generativeai returns .text for the combined output
    return getattr(resp, 'text', str(resp))

In [ ]:
answer = generate_with_gemini(rag_prompt)
answer